In [0]:
%pip install --quiet --upgrade databricks-vectorsearch flashrank

In [0]:
%restart_python

In [0]:
catalog = "workspace"
schema = "genai_course"

Creating Vector Search Endpoint

In [0]:
from databricks.vector_search.client import VectorSearchClient

vs_client = VectorSearchClient(disable_notice=True)
vs_endpoint_name = "vs_endpoint"

if vs_client.endpoint_exists(vs_endpoint_name):
    vs_client.wait_for_endpoint(vs_endpoint_name)
else:
    vs_client.create_endpoint(vs_endpoint_name, endpoint_type="STANDARD")


Setting up Delta Sync Index on our gold_table

In [0]:
gold_df = spark.read.table(f"{catalog}.{schema}.gold_docs_chunks")
display(gold_df)

In [0]:
source_table_fullname = f"{catalog}.{schema}.gold_docs_chunks"

spark.sql(f"ALTER TABLE {source_table_fullname} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")


Creating delta Sync Index Programatically

In [0]:
vs_index_fullname = f"{catalog}.{schema}.dphone_vs_index"

In [0]:
if vs_client.index_exists(vs_endpoint_name, vs_index_fullname):
    vs_index = vs_client.get_index(vs_endpoint_name, vs_index_fullname)
else:
    vs_index = vs_client.create_delta_sync_index_and_wait(
        source_table_name = source_table_fullname,
        index_name = vs_index_fullname,
        primary_key = "id",
        embedding_source_column = "chunk",
        embedding_model_endpoint_name = "databricks-gte-large-en",
        endpoint_name = vs_endpoint_name,
        pipeline_type = "TRIGGERED"
    )

    

Querying Programatically
Reranker is not available in free version of databricks

In [0]:
query_1 = "Explain the features of Dphone D1 Lite"

results = vs_index.similarity_search(
    query_text = query_1,
    columns = ["path", "chunk"],
    num_results = 3,
    query_type = "HYBRID"
  #  reranker = DatabricksReranker(columns_to_rerank, ["chunk"])
)

display(results)


In [0]:
passages = results.get("result", {}).get("data_array", {})
display(passages)